In [10]:
import pandas as pd

file_path = "data/CIDDS-001-Sampled-Data.parquet"
df = pd.read_parquet(file_path)

df.head()

,Date first seen,Duration,Proto,Src IP Addr,Src Pt,Dst IP Addr,Dst Pt,Packets,Bytes,Flows,Flags,Tos,class,attackType,attackID,attackDescription
0,2017-03-17 14:18:05.000,0.004,TCP,192.168.220.16,34242,192.168.100.6,80.0,6,545,1,.AP.SF,0,attacker,dos,18,10000 connections on 192.168.100.6:80
1,2017-03-17 14:18:05.000,0.003,TCP,192.168.220.16,34242,192.168.100.6,80.0,6,545,1,.AP.SF,0,attacker,dos,18,10000 connections on 192.168.100.6:80
2,2017-03-17 14:18:05.001,0.001,TCP,192.168.100.6,80,192.168.220.16,34242.0,4,272,1,.A..SF,0,victim,dos,18,10000 connections on 192.168.100.6:80
3,2017-03-17 14:18:05.001,0.003,TCP,192.168.100.6,80,192.168.220.16,34242.0,4,272,1,.A..SF,0,victim,dos,18,10000 connections on 192.168.100.6:80
4,2017-03-17 14:18:05.002,0.005,TCP,192.168.220.16,34243,192.168.100.6,80.0,6,545,1,.AP.SF,0,attacker,dos,18,10000 connections on 192.168.100.6:80


In [11]:
print(df.columns)

# Or as a list
print(list(df.columns))

Index(['Date first seen', 'Duration', 'Proto', 'Src IP Addr', 'Src Pt',
       'Dst IP Addr', 'Dst Pt', 'Packets', 'Bytes', 'Flows', 'Flags', 'Tos',
       'class', 'attackType', 'attackID', 'attackDescription'],
      dtype='str')
['Date first seen', 'Duration', 'Proto', 'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Packets', 'Bytes', 'Flows', 'Flags', 'Tos', 'class', 'attackType', 'attackID', 'attackDescription']


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, LabelEncoder

# Giả sử df là dữ liệu bạn đã load
# df = pd.read_parquet("data/CIDDS-001-Sampled-Data.parquet")

print("1. Đang làm sạch cột 'Bytes'...")
def clean_bytes(val):
    if pd.isna(val):
        return 0.0
    val = str(val).strip().upper()
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val:
        return float(val.replace('K', '')) * 1_000
    else:
        return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

# Cập nhật CHÍNH XÁC tên 10 cột đặc trưng dựa trên file thực tế của bạn
features = [
    'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 
    'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets'
]

# Tách features (X) và target (y)
X = df[features].copy()
y = df['attackType'].copy()

print("2. Đang chia tập Train/Test (Split 70/30)...")
# Stratify theo y để giữ nguyên tỷ lệ các loại tấn công
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("3. Đang mã hóa các biến phân loại (Categorical Encoding)...")
# Xác định các cột dạng chuỗi (thường là IP, Proto, Flags)
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# Fit TRÊN TẬP TRAIN để chống Data Leakage
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

# Chuyển toàn bộ X về float để chuẩn bị cho bước Scale
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# Mã hóa Label (attackType)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

print("4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...")
# Fit scaler CHỈ TRÊN TẬP TRAIN
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nHoàn tất! Kích thước dữ liệu:")
print(f"Train set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Các lớp tấn công đã encode: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

1. Đang làm sạch cột 'Bytes'...
2. Đang chia tập Train/Test (Split 70/30)...
3. Đang mã hóa các biến phân loại (Categorical Encoding)...
4. Đang chuẩn hóa dữ liệu (Min-Max Scaling 0-1)...

Hoàn tất! Kích thước dữ liệu:
Train set: (1833400, 10)
Test set: (785743, 10)
Các lớp tấn công đã encode: {'---': np.int64(0), 'bruteForce': np.int64(1), 'dos': np.int64(2), 'pingScan': np.int64(3), 'portScan': np.int64(4)}


In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score

# 1. MÃ HÓA LẠI THEO TẦN SUẤT (FREQUENCY ENCODING) CHO KNN VÀ RF
# (Chạy lại từ bước tách X, y)
X = df[features].copy()
y = df['attackType'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# Dùng Frequency Encoding: Thay giá trị chuỗi bằng tỷ lệ xuất hiện của nó
# Cách này giữ nguyên được 10 features mà vẫn thể hiện được bản chất của dữ liệu mạng
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0) # Điền 0 nếu tập test có nhãn mới
    X_test[col] = X_test[col].map(freq).fillna(0)

# Chuyển và chuẩn hóa
X_train = X_train.astype(float)
X_test = X_test.astype(float)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. CHẠY LẠI MÔ HÌNH VỚI THAM SỐ TINH CHỈNH
print("--- ĐANG TRAIN LẠI MÔ HÌNH ---")

# Đổi max_features='sqrt' cho giống tính chất của thuật toán phân loại RF
rf_model_tuned = RandomForestClassifier(
    n_estimators=10,             
    criterion='gini',            
    min_samples_split=2,         
    min_samples_leaf=1,          
    max_features='sqrt',         # Đã tinh chỉnh ở đây
    min_impurity_decrease=0.0,   
    class_weight=None,     
    random_state=42,             
    n_jobs=-1                    
)
rf_model_tuned.fit(X_train_scaled, y_train)
rf_preds_tuned = rf_model_tuned.predict(X_test_scaled)

# KNN giữ nguyên, nhưng nó sẽ học tốt hơn nhiều nhờ Frequency Encoding
knn_model_tuned = KNeighborsClassifier(
    n_neighbors=3, weights='uniform', leaf_size=30, metric='minkowski', n_jobs=-1
)
knn_model_tuned.fit(X_train_scaled, y_train)
knn_preds_tuned = knn_model_tuned.predict(X_test_scaled)

print("\n--- KẾT QUẢ SAU KHI TINH CHỈNH (MACRO F1) ---")
print(f"Random Forest F1-Score: {f1_score(y_test, rf_preds_tuned, average='macro'):.4f}")
print(f"KNN F1-Score:           {f1_score(y_test, knn_preds_tuned, average='macro'):.4f}")

print("\n--- CHI TIẾT LỚP THIỂU SỐ TỪ KẾT QUẢ RANDOM FOREST ---")
print(classification_report(y_test, rf_preds_tuned, target_names=label_encoder.classes_, digits=4, zero_division=0))

--- ĐANG TRAIN LẠI MÔ HÌNH ---

--- KẾT QUẢ SAU KHI TINH CHỈNH (MACRO F1) ---
Random Forest F1-Score: 0.9275
KNN F1-Score:           0.9235

--- CHI TIẾT LỚP THIỂU SỐ TỪ KẾT QUẢ RANDOM FOREST ---
              precision    recall  f1-score   support

         ---     0.9999    1.0000    1.0000    652723
  bruteForce     0.9892    0.9710    0.9800       379
         dos     1.0000    0.9999    1.0000    117280
    pingScan     0.8108    0.5625    0.6642       320
    portScan     0.9908    0.9958    0.9933     15041

    accuracy                         0.9997    785743
   macro avg     0.9582    0.9058    0.9275    785743
weighted avg     0.9997    0.9997    0.9997    785743



In [ ]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

print("======================================================")
print("   BẮT ĐẦU CHẠY FULL DATA: CHỈ TEST RANDOM FOREST     ")
print("======================================================")

# ==========================================
# 1. LOAD VÀ TIỀN XỬ LÝ CHUNG
# ==========================================
file_path = "data/CIDDS-001-Sampled-Data.parquet"
df = pd.read_parquet(file_path)
print(f"-> Đã nạp thành công {len(df)} dòng dữ liệu gốc.")

def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    else: return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

features = ['Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['attackType'].copy()

del df
gc.collect()

# ==========================================
# 2. CHIA TẬP TRAIN/TEST VÀ CHUẨN HÓA
# ==========================================
print("\n-> Đang chia tập Train/Test theo thời gian...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

print("-> Đang thực hiện Frequency Encoding và Min-Max Scaling...")
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

X_train, X_test = X_train.astype(np.float32), X_test.astype(np.float32) 
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# ==========================================
# 3. TẠO CHUỖI MULTI-FLOW TỐI ƯU
# ==========================================
# BẠN CÓ THỂ ĐỔI THÀNH 70 Ở ĐÂY ĐỂ KIỂM CHỨNG SỰ TỤT DỐC CỦA RF
window_rf = 10 
print(f"\n-> Đang tạo ma trận chuỗi 3D cho RF (Window = {window_rf})...")

def create_sequences_optimized(X_data, y_data, window_size):
    n_samples = len(X_data) - window_size + 1
    X_seq = np.zeros((n_samples, window_size, X_data.shape[1]), dtype=np.float32)
    y_seq = np.zeros((n_samples,), dtype=y_data.dtype)
    for i in range(window_size - 1, len(X_data)):
        idx = i - window_size + 1
        X_seq[idx] = X_data[idx : i + 1] 
        y_seq[idx] = y_data[i]           
    return X_seq, y_seq

X_train_seq_rf, y_train_seq_rf = create_sequences_optimized(X_train_scaled, y_train_encoded, window_rf)
X_test_seq_rf, y_test_seq_rf = create_sequences_optimized(X_test_scaled, y_test_encoded, window_rf)

# ==========================================
# 4. HUẤN LUYỆN MÔ HÌNH RANDOM FOREST
# ==========================================
print("\n======================================================")
print(f"   HUẤN LUYỆN RANDOM FOREST (LÀM PHẲNG WINDOW = {window_rf})")
print("======================================================")

# Làm phẳng mảng 3D thành 2D để đưa vào RF
X_train_flat = X_train_seq_rf.reshape(X_train_seq_rf.shape[0], -1)
X_test_flat = X_test_seq_rf.reshape(X_test_seq_rf.shape[0], -1)

# Cấu hình RF chuẩn theo Paper 2
rf_multi = RandomForestClassifier(
    n_estimators=100, 
    max_depth=35, 
    max_features=100, 
    class_weight='balanced', 
    n_jobs=-1, 
    random_state=42
)

print("-> Đang huấn luyện Random Forest... (Vui lòng chờ vài phút)")
rf_multi.fit(X_train_flat, y_train_seq_rf)

print("-> Đang dự đoán tập Test...")
rf_multi_preds = rf_multi.predict(X_test_flat)

# Tìm các nhãn thực sự có trong tập test của chuỗi
labels_in_test_rf = np.unique(y_test_seq_rf)
names_in_test_rf = label_encoder.inverse_transform(labels_in_test_rf)

print(f"\n=> KẾT QUẢ: F1-Score Random Forest (Window {window_rf}): {f1_score(y_test_seq_rf, rf_multi_preds, average='macro'):.4f}")
print(f"\n[BẢNG CHI TIẾT TỪNG LỚP TẤN CÔNG (WINDOW {window_rf})]")
print(classification_report(y_test_seq_rf, rf_multi_preds, labels=labels_in_test_rf, target_names=names_in_test_rf, digits=4, zero_division=0))

   BẮT ĐẦU CHẠY FULL DATA: CHỈ TEST RANDOM FOREST     
-> Đã nạp thành công 2619143 dòng dữ liệu gốc.

-> Đang chia tập Train/Test theo thời gian...
-> Đang thực hiện Frequency Encoding và Min-Max Scaling...

-> Đang tạo ma trận chuỗi 3D cho RF (Window = 10)...

   HUẤN LUYỆN RANDOM FOREST (LÀM PHẲNG WINDOW = 10)
-> Đang huấn luyện Random Forest... (Vui lòng chờ vài phút)
-> Đang dự đoán tập Test...

=> KẾT QUẢ: F1-Score Random Forest (Window 10): 0.5048

[BẢNG CHI TIẾT TỪNG LỚP TẤN CÔNG (WINDOW 10)]
              precision    recall  f1-score   support

         ---     0.9779    0.9999    0.9888    618597
  bruteForce     0.3333    0.0026    0.0052       381
         dos     0.9998    0.9997    0.9997    144845
    pingScan     0.0094    0.0098    0.0096       307
    portScan     0.9607    0.3571    0.5207     21604

    accuracy                         0.9813    785734
   macro avg     0.6562    0.4738    0.5048    785734
weighted avg     0.9808    0.9813    0.9771    785734



In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("--- KHỞI ĐỘNG PHIÊN BẢN SPEEDTEST (100K DÒNG) ---")
file_path = "data/CIDDS-001-Sampled-Data.parquet"
df_full = pd.read_parquet(file_path)

# CẮT NHỎ DỮ LIỆU ĐỂ TEST NHANH (Bảo toàn trình tự thời gian)
df = df_full.iloc[:100000].copy() 
print(f"Đã cắt dữ liệu xuống còn {len(df)} dòng để test nhanh!")

# ==========================================
# 1. TIỀN XỬ LÝ LÀM SẠCH
# ==========================================
def clean_bytes(val):
    if pd.isna(val): return 0.0
    val = str(val).strip().upper()
    if 'M' in val: return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val: return float(val.replace('K', '')) * 1_000
    else: return float(val)

df['Bytes'] = df['Bytes'].apply(clean_bytes)

features = ['Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 'Flags', 'Tos', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['attackType'].copy()

# ==========================================
# 2. CHIA TẬP TRAIN/TEST (CHỐNG DATA LEAKAGE)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# ==========================================
# 3. MÃ HÓA VÀ CHUẨN HÓA AN TOÀN
# ==========================================
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0)
    X_test[col] = X_test[col].map(freq).fillna(0)

X_train, X_test = X_train.astype(float), X_test.astype(float)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
num_classes = len(label_encoder.classes_)

# ==========================================
# 4. TẠO CHUỖI MULTI-FLOW ĐỘC LẬP
# ==========================================
def create_sequences(X_data, y_data, window_size=10):
    X_seq, y_seq = [], []
    for i in range(len(X_data) - window_size):
        X_seq.append(X_data[i : i + window_size])
        y_seq.append(y_data[i + window_size])
    return np.array(X_seq), np.array(y_seq)

window_size = 10
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_encoded, window_size)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_encoded, window_size)

y_train_categorical = tf.keras.utils.to_categorical(y_train_seq, num_classes=num_classes)
y_test_categorical = tf.keras.utils.to_categorical(y_test_seq, num_classes=num_classes)

# --- GIẢI QUYẾT LỖI IN KẾT QUẢ KHI TEST TẬP NHỎ ---
# Xác định chính xác các nhãn tồn tại trong tập Test để report không bị báo lỗi thiếu nhãn
labels_in_test = np.unique(y_test_seq)
names_in_test = label_encoder.inverse_transform(labels_in_test)

# ==========================================
# 5. HUẤN LUYỆN RANDOM FOREST
# ==========================================
print("\n--- HUẤN LUYỆN RANDOM FOREST ---")
X_train_flat = X_train_seq.reshape(X_train_seq.shape[0], -1)
X_test_flat = X_test_seq.reshape(X_test_seq.shape[0], -1)

rf_multi = RandomForestClassifier(n_estimators=50, max_depth=20, max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42)
rf_multi.fit(X_train_flat, y_train_seq)
rf_multi_preds = rf_multi.predict(X_test_flat)

print(f"-> F1-Score Random Forest (Multi-flow): {f1_score(y_test_seq, rf_multi_preds, average='macro'):.4f}")
print("\n[CHI TIẾT RANDOM FOREST]")
print(classification_report(y_test_seq, rf_multi_preds, labels=labels_in_test, target_names=names_in_test, digits=4, zero_division=0))

# ==========================================
# 6. HUẤN LUYỆN LSTM VỚI CLASS WEIGHT
# ==========================================
print("\n--- HUẤN LUYỆN LSTM VỚI CLASS WEIGHT ---")
unique_classes = np.unique(y_train_seq)
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=y_train_seq)
class_weight_dict = dict(zip(unique_classes, weights))
print(f"Trọng số cân bằng được gán: {class_weight_dict}")

lstm_model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(window_size, X_train_seq.shape[2]), activation='tanh'),
    Dropout(0.2),
    LSTM(100, return_sequences=False, activation='tanh'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

lstm_model.fit(
    X_train_seq, y_train_categorical, 
    epochs=5, 
    batch_size=512, 
    validation_split=0.1, 
    class_weight=class_weight_dict, 
    verbose=1
)

lstm_preds_probs = lstm_model.predict(X_test_seq)
lstm_preds = np.argmax(lstm_preds_probs, axis=1)

print(f"\n-> F1-Score LSTM (Multi-flow): {f1_score(y_test_seq, lstm_preds, average='macro'):.4f}")
print("\n[CHI TIẾT LSTM SAU KHI FIX TRỌNG SỐ]")
print(classification_report(y_test_seq, lstm_preds, labels=labels_in_test, target_names=names_in_test, digits=4, zero_division=0))

--- KHỞI ĐỘNG PHIÊN BẢN SPEEDTEST (100K DÒNG) ---
Đã cắt dữ liệu xuống còn 100000 dòng để test nhanh!

--- HUẤN LUYỆN RANDOM FOREST ---
-> F1-Score Random Forest (Multi-flow): 1.0000

--- HUẤN LUYỆN LSTM VỚI CLASS WEIGHT ---
Trọng số cân bằng được gán: {np.int64(0): np.float64(4.679727199786039), np.int64(1): np.float64(0.5598125159969286)}
Epoch 1/5


/home/luongminhthu/Documents/code/university/AI66A_CANHAN2/venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


124/124 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.9725 - loss: 0.2442 - val_accuracy: 0.9940 - val_loss: 0.1653
Epoch 2/5
124/124 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.9769 - loss: 0.2244 - val_accuracy: 0.9939 - val_loss: 0.1407
Epoch 3/5
124/124 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.9754 - loss: 0.2150 - val_accuracy: 0.9937 - val_loss: 0.0912
Epoch 4/5
124/124 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.9749 - loss: 0.2152 - val_accuracy: 0.9936 - val_loss: 0.0933
Epoch 5/5
124/124 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9727 - loss: 0.2095 - val_accuracy: 0.9931 - val_loss: 0.1160
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step

-> F1-Score LSTM (Multi-flow): 1.0000

[CHI TIẾT LSTM SAU KHI FIX TRỌNG SỐ]


ValueError: Number of classes, 1, does not match size of target_names, 2. Try specifying the labels parameter